# Stage 1 — Exploratory Data Analysis
**Student Performance Analysis** | Dataset: Yılmaz & Şekeroğlu (2019), UCI ML Repository

Goal: understand the target distribution, identify the strongest associations with `grade`, check for confounds and multicollinearity, and surface the decisions that should carry into preprocessing/modeling. See `reports/eda_findings.md` for the full written analysis — this notebook is the reproducible source.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr, kruskal, mannwhitneyu
from src.data.load_data import load_raw

sns.set_theme(style='whitegrid')
df = load_raw()
df.shape

## 1. Target distribution

In [ ]:
grade_labels = {0:'Fail',1:'DD',2:'DC',3:'CC',4:'CB',5:'BB',6:'BA',7:'AA'}
counts = df['grade'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7,4.5))
sns.barplot(x=[grade_labels[i] for i in counts.index], y=counts.values, ax=ax)
ax.set_xlabel('Grade'); ax.set_ylabel('Number of students')
ax.set_title('Target distribution: end-of-term GRADE (n=145)')
for i, v in enumerate(counts.values):
    ax.text(i, v+0.5, str(v), ha='center')
plt.tight_layout(); plt.show()
counts

**Observation:** moderately imbalanced (Fail n=8 vs DD n=35). Accuracy alone is not a sufficient metric, macro-F1 will be used as the primary metric going forward.

## 2. Association of every predictor with the target
Cramér's V (nominal association, any monotonic or non-monotonic relationship) and Spearman ρ (captures ordinal/monotonic relationship, matches the natural ordering of `grade`).

In [ ]:
def cramers_v(x, y):
    confusion = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    denom = min((kcorr-1), (rcorr-1))
    return np.sqrt(phi2corr / denom) if denom > 0 else np.nan

predictors = [c for c in df.columns if c not in ('student_id', 'grade')]
rows = []
for col in predictors:
    cv = cramers_v(df[col], df['grade'])
    rho, p = spearmanr(df[col], df['grade'])
    rows.append({'feature': col, 'cramers_v': round(cv,3), 'spearman_rho': round(rho,3), 'spearman_p': round(p,4)})
assoc = pd.DataFrame(rows).sort_values('cramers_v', ascending=False).reset_index(drop=True)
assoc

## 3. Course-level effect
Course identity shows the strongest nominal association with grade. Is this difference statistically significant?

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
order = df.groupby('course_id')['grade'].median().sort_values().index
sns.boxplot(data=df, x='course_id', y='grade', order=order, ax=ax)
sns.stripplot(data=df, x='course_id', y='grade', order=order, ax=ax, color='black', size=3, alpha=0.4)
ax.set_title('Grade by course (course 1 alone has n=66; course 2 only n=2)')
plt.tight_layout(); plt.show()

groups = [g['grade'].values for _, g in df.groupby('course_id')]
stat, p = kruskal(*groups)
print(f'Kruskal-Wallis H={stat:.2f}, p={p:.2e}')

**Observation:** Highly significant (p < 0.00001). Course is the single strongest predictor available, this will be an explicit modeling decision (include `course_id` vs. analyze within-course).

## 4. Sex effect and its confound with course
Sex shows the second-strongest marginal association. Before accepting that at face value, it will be checked whether sex is evenly distributed across courses.

In [ ]:
m = df[df.sex==2]['grade']; f = df[df.sex==1]['grade']
u, p_sex = mannwhitneyu(m, f)
print(f'Pooled Mann-Whitney U test: U={u:.1f}, p={p_sex:.2e}  (male n={len(m)}, female n={len(f)})')

ct = pd.crosstab(df['course_id'], df['sex'])
ct.columns = ['Female','Male']
ct['pct_female'] = (ct['Female']/(ct['Female']+ct['Male'])*100).round(0)
ct['mean_grade'] = df.groupby('course_id')['grade'].mean().round(2)
ct

**Critical finding:** Course 9 (n=21, mean grade 2.19) is 100% female; course 8 (n=14, mean grade 1.36) is 79% female. These are 2 of the 3 lowest-scoring courses. The pooled sex effect is substantially confounded by course composition. Within course 1 alone (the one course with a usable n for both sexes), the gap narrows considerably. **This will be reported with the confound made explicit 'sex predicts grade' finding.**

In [ ]:
within_course1 = df[df.course_id==1].groupby('sex')['grade'].agg(['mean','count'])
within_course1.index = ['Female','Male']
within_course1

## 5. Prior performance (GPA) and multicollinearity

In [ ]:
fig, ax = plt.subplots(figsize=(6,4.5))
sns.boxplot(data=df, x='gpa_last_semester', y='grade', ax=ax)
ax.set_xlabel('Prior-semester GPA bracket (1=<2.00 ... 5=>3.49)')
ax.set_title('Grade vs prior-semester GPA bracket')
plt.tight_layout(); plt.show()

corr = df[predictors].corr(method='spearman')
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print('Predictor pairs with |Spearman rho| > 0.5:')
print(corr_pairs[corr_pairs.abs() > 0.5].sort_values(key=abs, ascending=False))

**Observation:** `gpa_last_semester` is the most defensible *behavioral* (non-confounded) predictor, plausibly a genuine causal precursor. It correlates moderately with `gpa_expected_graduation` (ρ=0.65), the only multicollinear pair in the dataset, a decision to make explicitly in Stage 2.

## Summary: decisions carried into Stage 2
1. Primary metric: **macro-F1**, with accuracy reported alongside.
2. **Stratified** train/test or k-fold splits, given imbalance and small n.
3. Model **with and without `course_id`** as separate.
4. `Sex` will be kept as a feature but the course confound will alwalys report alongside any sex-related finding.
5. Resolve the `gpa_last_semester` / `gpa_expected_graduation` multicollinearity explicitly (keep both + regularize)